<a href="https://colab.research.google.com/github/GottaLearnItALL/Machine_Learning_Final_Project/blob/main/ML_final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!uv run main.py

  PEPTIDE TOXICITY PREDICTION PIPELINE
  CUDA available: Tesla T4

[STEP 1] Loading data...
  Parsed Train.fasta: 6387 sequences (toxic=4569, non-toxic=1818)
  Parsed test.fasta: 1126 sequences (toxic=806, non-toxic=320)

  Training: 6387 samples | Test: 1126 samples
  Train labels: {0: np.int64(4569), 1: np.int64(1818)}
  Test labels:  {0: np.int64(806), 1: np.int64(320)}

[STEP 2] Extracting features...

  --- Physicochemical ---
  Physicochemical features: 31 features extracted
  Physicochemical features: 31 features extracted

  --- Sequence-Based ---
  Sequence features: 420 features (20 AAC + 400 DPC)
  Sequence features: 420 features (20 AAC + 400 DPC)

  --- PLM Embeddings ---
  Loading PLM: facebook/esm2_t6_8M_UR50D...
config.json: 100% 775/775 [00:00<00:00, 4.00MB/s]
tokenizer_config.json: 100% 95.0/95.0 [00:00<00:00, 412kB/s]
vocab.txt: 100% 93.0/93.0 [00:00<00:00, 498kB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 576kB/s]
model.safetensors: 100% 31.4M/31.4M [00:0

In [13]:
import pandas as pd
results_df = pd.read_csv('results/all_results.csv')  # adjust path if needed

In [19]:
import warnings
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
import config
from parse_fasta import load_data
import features_physicochemical, features_sequence, features_plm

warnings.filterwarnings('ignore')
np.random.seed(config.RANDOM_STATE)

train_df, test_df = load_data(config.TRAIN_FILE, config.TEST_FILE)
train_seqs = train_df['sequence'].tolist()
test_seqs  = test_df['sequence'].tolist()
y_train    = train_df['label'].values
y_test     = test_df['label'].values

physchem_train = features_physicochemical.extract(train_seqs)
physchem_test  = features_physicochemical.extract(test_seqs)
seq_train      = features_sequence.extract(train_seqs)
seq_test       = features_sequence.extract(test_seqs)
plm_train      = features_plm.extract(train_seqs)
plm_test       = features_plm.extract(test_seqs)
print("features ready")

  Parsed Train.fasta: 6387 sequences (toxic=4569, non-toxic=1818)
  Parsed test.fasta: 1126 sequences (toxic=806, non-toxic=320)
  Physicochemical features: 31 features extracted
  Physicochemical features: 31 features extracted
  Sequence features: 420 features (20 AAC + 400 DPC)
  Sequence features: 420 features (20 AAC + 400 DPC)
  Loading PLM: facebook/esm2_t6_8M_UR50D...


Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  PLM running on: cuda
    Processed 100/6387 sequences...
    Processed 200/6387 sequences...
    Processed 300/6387 sequences...
    Processed 400/6387 sequences...
    Processed 500/6387 sequences...
    Processed 600/6387 sequences...
    Processed 700/6387 sequences...
    Processed 800/6387 sequences...
    Processed 900/6387 sequences...
    Processed 1000/6387 sequences...
    Processed 1100/6387 sequences...
    Processed 1200/6387 sequences...
    Processed 1300/6387 sequences...
    Processed 1400/6387 sequences...
    Processed 1500/6387 sequences...
    Processed 1600/6387 sequences...
    Processed 1700/6387 sequences...
    Processed 1800/6387 sequences...
    Processed 1900/6387 sequences...
    Processed 2000/6387 sequences...
    Processed 2100/6387 sequences...
    Processed 2200/6387 sequences...
    Processed 2300/6387 sequences...
    Processed 2400/6387 sequences...
    Processed 2500/6387 sequences...
    Processed 2600/6387 sequences...
    Processed 2700/6387 

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  PLM running on: cuda
    Processed 100/1126 sequences...
    Processed 200/1126 sequences...
    Processed 300/1126 sequences...
    Processed 400/1126 sequences...
    Processed 500/1126 sequences...
    Processed 600/1126 sequences...
    Processed 700/1126 sequences...
    Processed 800/1126 sequences...
    Processed 900/1126 sequences...
    Processed 1000/1126 sequences...
    Processed 1100/1126 sequences...
  PLM features: 320 dimensions from ESM-2
features ready


In [20]:
import model_fusion

views_train_scaled, views_test_scaled = [], []
for ft_train, ft_test in [(physchem_train, physchem_test),
                           (seq_train, seq_test),
                           (plm_train, plm_test)]:
    sc = StandardScaler()
    views_train_scaled.append(sc.fit_transform(ft_train.values))
    views_test_scaled.append(sc.transform(ft_test.values))

fusion_result, fusion_model, attn_weights = model_fusion.train_and_evaluate(
    views_train_scaled, views_test_scaled, y_train, y_test
)

# save it this time
np.save('results/attn_weights.npy', attn_weights)
print("attn_weights saved!")

    Fusion device: cuda
      Epoch 10/50 | Loss: 0.0464
      Epoch 20/50 | Loss: 0.0266
      Epoch 30/50 | Loss: 0.0184
      Epoch 40/50 | Loss: 0.0170
      Epoch 50/50 | Loss: 0.0207

  Attention Fusion Results:
    Acc=0.9121 | Sen=0.9404 | Spe=0.8406 | MCC=0.7833
  Average Attention Weights:
    Physicochemical: 0.0764 (7.6%)
    Sequence-Based: 0.5301 (53.0%)
    PLM: 0.3936 (39.4%)
attn_weights saved!


In [27]:
import importlib, visualize
importlib.reload(visualize)

results_df = pd.read_csv('results/all_results.csv')
visualize.generate_all_plots(results_df, attn_weights)
print("done!")


[VISUALIZATIONS]
  Saved: results/classifier_accuracy.png
  Saved: results/classifier_mcc.png
  Saved: results/feature_group_comparison.png
  Saved: results/attention_weights.png
done!
